### 1. 데이터 준비

In [1]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ta
from sklearn.preprocessing import MinMaxScaler

In [2]:
tickers = ['AMZN']

start_date = '2019-01-01'
end_date = '2024-09-30'

In [3]:
# 데이터 준비 함수
def prepare_stock_data(ticker, start_date, end_date):
    stock_df = yf.download(ticker, start=start_date, end=end_date)

    stock_df.columns = [col[0] for col in stock_df.columns]

    # Open, Close, High, Low를 Adj Close 기준으로 스케일링
    stock_df['Open_Scaled'] = stock_df['Open'] / stock_df['Adj Close']
    stock_df['Close_Scaled'] = stock_df['Close'] / stock_df['Adj Close']
    stock_df['High_Scaled'] = stock_df['High'] / stock_df['Adj Close']
    stock_df['Low_Scaled'] = stock_df['Low'] / stock_df['Adj Close']

    # 시가-종가 차이
    stock_df['OC_Change'] = (stock_df['Close'] - stock_df['Open']) / stock_df['Adj Close']

    # 필요하지 않은 절대값 열 제거
    stock_df.drop(['Open', 'Close', 'High', 'Low'], axis=1, inplace=True)

    # 기술적 지표 추가
    adj_close_series = stock_df['Adj Close']
    volume_series = stock_df['Volume']

    ## 1.1. 단순 이동평균 (Simple Moving Average)
    stock_df['MA5'] = adj_close_series.rolling(window=5).mean()
    stock_df['MA20'] = adj_close_series.rolling(window=20).mean()
    stock_df['MA60'] = adj_close_series.rolling(window=60).mean()
    stock_df['MA120'] = adj_close_series.rolling(window=120).mean()

    ## 1.2. 지수 이동 평균 (Exponential Moving Average)
    stock_df['EMA5'] = adj_close_series.ewm(span=5, adjust=False).mean()
    stock_df['EMA20'] = adj_close_series.ewm(span=20, adjust=False).mean()
    stock_df['EMA60'] = adj_close_series.ewm(span=60, adjust=False).mean()
    stock_df['EMA120'] = adj_close_series.ewm(span=120, adjust=False).mean()

    ## 1.3. 더블 볼린저 밴드 (Double Bollinger Bands indicator)
    adj_close_series_1d = adj_close_series.squeeze()
    stock_df['BOL_AVG'] = ta.volatility.bollinger_mavg(adj_close_series_1d)
    rolling_std = adj_close_series_1d.rolling(window=20).std()
    stock_df['BOL_H1'] = stock_df['BOL_AVG'] + 2 * rolling_std
    stock_df['BOL_L1'] = stock_df['BOL_AVG'] - 2 * rolling_std
    stock_df['BOL_H2'] = stock_df['BOL_AVG'] + rolling_std
    stock_df['BOL_L2'] = stock_df['BOL_AVG'] - rolling_std

    ## 1.4. RSI(Relative Strength Index)
    stock_df['RSI'] = ta.momentum.rsi(adj_close_series_1d)

    ## 1.5. MACD (Moving Average Convergence Divergence)
    stock_df['MACD'] = ta.trend.macd(adj_close_series_1d)
    stock_df['MACD_SIGNAL'] = ta.trend.macd_signal(adj_close_series_1d)

    ## 1.6. OBV (On-Balance Volume)
    volume_series_1d = volume_series.squeeze()
    stock_df['OBV'] = ta.volume.on_balance_volume(adj_close_series_1d, volume_series_1d)

    ## Target 설정
    stock_df['Change'] = stock_df['Adj Close'].pct_change()  # 변동률 계산
    stock_df['Target'] = stock_df['Change'].apply(lambda x: 1 if x > 0.01 else 2 if x < -0.01 else 0)

    # 공휴일 또는 주말: 전날 값으로 채움
    stock_df.fillna(method='ffill', inplace=True)

    # 이동평균 등으로 인해 남아 있는 NaN 값 제거
    stock_df.dropna(inplace=True)

    features_to_scale = ['Adj Close', 'Volume', 'MA5', 'MA20', 'MA60', 'MA120',
                         'EMA5', 'EMA20', 'EMA60', 'EMA120', 'BOL_AVG', 'RSI', 'MACD', 'MACD_SIGNAL', 'OBV']

    scaler = MinMaxScaler()
    stock_df[features_to_scale] = scaler.fit_transform(stock_df[features_to_scale])
    
    return stock_df, scaler  # 데이터프레임과 스케일러 반환

In [4]:
scalers = {}

for ticker in tickers:
    df, scaler = prepare_stock_data(ticker, start_date, end_date)  # scaler 반환
    globals()[f"{ticker}_df"] = df  # 준비된 데이터를 globals에 저장
    scalers[ticker] = scaler 

[*********************100%***********************]  1 of 1 completed
C:\TempFolder\ipykernel_7740\3880662535.py:60: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  stock_df.fillna(method='ffill', inplace=True)


In [5]:
AMZN_df.head(3)

,Adj Close,Volume,Open_Scaled,Close_Scaled,High_Scaled,Low_Scaled,OC_Change,MA5,MA20,MA60,...,BOL_H1,BOL_L1,BOL_H2,BOL_L2,RSI,MACD,MACD_SIGNAL,OBV,Change,Target
Date,,,,,,,,,,,,,,,,,,,,,
2019-06-24 00:00:00+00:00,0.117406,0.095445,0.999352,1.0,1.001547,0.993417,0.000648,0.103085,0.051162,0.050096,...,98.621551,85.005199,95.217463,88.409287,0.676353,0.653255,0.627336,0.156848,0.001360,0
2019-06-25 00:00:00+00:00,0.102331,0.145104,1.017873,1.0,1.020295,0.996885,-0.017873,0.101087,0.052138,0.050916,...,98.790653,85.045297,95.354314,88.481636,0.538037,0.649225,0.633433,0.142903,-0.018616,2
2019-06-26 00:00:00+00:00,0.110607,0.106264,0.997181,1.0,1.003146,0.994462,0.002819,0.100139,0.053970,0.051619,...,99.095907,85.133242,95.605241,88.623908,0.594985,0.649351,0.638339,0.154207,0.010414,1


### 2. LSTM 수행

In [ ]:
from sklearn.model_selection import ParameterGrid
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import accuracy_score

In [ ]:
# 데이터 분리 함수
def split_data(data, window_size, test_year=1):
    train_data = data[:-252]  # 최근 1년 제외
    test_data = data[-252:]  # 최근 1년
    
    def create_sequences(data, window_size):
        X, y = [], []
        for i in range(len(data) - window_size):
            X.append(data.iloc[i:i + window_size, :-1].values)  # Feature columns
            y.append(data.iloc[i + window_size, -1])  # Target column
        return np.array(X), np.array(y)
    
    X_train, y_train = create_sequences(train_data, window_size)
    X_test, y_test = create_sequences(test_data, window_size)
    
    return X_train, y_train, X_test, y_test

In [ ]:
# 모델 빌드 함수
def build_lstm_model(input_shape, units, dropout_rate, learning_rate):
    optimizer = Adam(learning_rate=float(learning_rate))
    model = Sequential([
        Input(shape=input_shape),
        LSTM(units, return_sequences=False),
        Dropout(dropout_rate),
        Dense(3, activation='softmax')  # 3 클래스 (Up, Down, Hold)
    ])
    model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def train_model(model, X_train, y_train, batch_size, epochs):
    model.fit(X_train, y_train, batch_size=batch_size, epochs=epochs, verbose=0)
    return model

In [ ]:
# 모델 훈련 및 평가
def train_and_evaluate_lstm(stock_df, param_grid):
    results = []
    grid = ParameterGrid(param_grid)
    for params in grid:
        window_size = params['window_size']
        X_train, y_train, X_test, y_test = split_data(stock_df, window_size)
        
        # 새로운 LSTM 모델 생성
        model = build_lstm_model(
            input_shape=(window_size, X_train.shape[2]),
            units=params['units'],
            dropout_rate=params['dropout_rate'],
            learning_rate=float(params['optimizer'].learning_rate.numpy())
        )
        
        # @tf.function 사용하여 retracing 문제 해결
        train_model(model, X_train, y_train, params['batch_size'], params['epochs'])
        
        y_pred = np.argmax(model.predict(X_test), axis=1)
        
        accuracy = accuracy_score(y_test, y_pred)
        total_return = calculate_total_return(y_test, y_pred, stock_df[-252:])  # Custom function
        
        results.append({
            'params': params,
            'accuracy': accuracy,
            'total_return': total_return
        })
    
    # 최적 파라미터 및 모델 선택
    best_result = max(results, key=lambda x: x['total_return'])
    best_model = build_lstm_model(
        input_shape=(best_result['params']['window_size'], X_train.shape[2]),
        units=best_result['params']['units'],
        dropout_rate=best_result['params']['dropout_rate'],
        learning_rate=float(best_result['params']['optimizer'].learning_rate.numpy())
    )
    train_model(best_model, X_train, y_train, best_result['params']['batch_size'], best_result['params']['epochs'])
    
    return best_model, best_result

In [ ]:
# 수익률 계산 함수
def calculate_total_return(y_test, y_pred, test_data):
    # test_data['Adj Close']와 예측 값 기반으로 수익률 계산
    initial_cash = 10000  # 초기 자본
    cash = initial_cash
    for i in range(len(y_test)):
        if y_pred[i] == 1:  # 매수
            cash *= (1 + test_data.iloc[i + 1]['Change'])
        elif y_pred[i] == 2:  # 매도
            cash *= (1 - test_data.iloc[i + 1]['Change'])
    return (cash - initial_cash) / initial_cash

In [ ]:
# 그리드 서치 설정
param_grid = {
    'units': [32, 64, 128],
    'dropout_rate': [0.2, 0.3, 0.4],
    'batch_size': [32, 64],
    'epochs': [20, 30, 50],
    'optimizer': [Adam(learning_rate=0.001)],
    'window_size': [30, 60]
}

In [ ]:
# 실행
best_models = {}
for ticker in tickers:
    stock_df = globals()[f"{ticker}_df"]
    best_model, best_result = train_and_evaluate_lstm(stock_df, param_grid)
    best_models[ticker] = (best_model, best_result)

    print(f"Ticker: {ticker}")
    print("Best Parameters:", best_result['params'])
    print("Best Accuracy:", best_result['accuracy'])
    print("Total Return:", best_result['total_return'])

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step  
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step  
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step  
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step  
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step  
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step


KeyboardInterrupt: 

### 3. 성능 평가

In [ ]:
def visualize_accuracy(test_df, y_pred):
    """
    매수/매도 신호 정확도 계산 및 시각화.

    Parameters:
        test_df (DataFrame): 테스트 데이터프레임 (Target 열 필요).
        y_pred (array): 모델의 예측 결과.
    """
    # 실제 값과 예측 값 준비
    y_true = test_df['Target'][-len(y_pred):]  # 테스트 데이터의 실제 Target
    y_pred = np.array(y_pred)  # 모델 예측 결과
    
    # 매수(1) 및 매도(2) 신호 정확도 계산
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    print("\nClassification Report:\n", classification_report(y_true, y_pred, labels=[0, 1, 2]))

    # Confusion Matrix 시각화
    plt.figure(figsize=(8, 6))
    plt.matshow(cm, cmap='Blues', fignum=1)
    plt.colorbar()
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix")
    plt.xticks([0, 1, 2], ["Hold (0)", "Buy (1)", "Sell (2)"])
    plt.yticks([0, 1, 2], ["Hold (0)", "Buy (1)", "Sell (2)"])

    for (i, j), val in np.ndenumerate(cm):
        plt.text(j, i, f'{val}', ha='center', va='center')

    plt.show()


# 모델 예측 (예시, 실제 모델에 따라 변경)
y_test_true = test_df['Target'].values  # 테스트 데이터의 실제 Target 값
y_test_pred = np.random.choice([0, 1, 2], size=len(y_test_true))  # 랜덤 예측 (모델 예측 결과로 대체)

# 정확도 시각화
visualize_accuracy(test_df, y_test_pred)


In [ ]:
def visualize_accuracy(test_df, y_pred):
    """
    매수/매도 신호 정확도 계산 및 시각화.

    Parameters:
        test_df (DataFrame): 테스트 데이터프레임 (Target 열 필요).
        y_pred (array): 모델의 예측 결과.
    """
    # 실제 값과 예측 값 준비
    y_true = test_df['Target'].values[-len(y_pred):]  # 테스트 데이터의 실제 Target
    y_pred = np.array(y_pred)  # 모델 예측 결과
    
    # 매수(1), 매도(2), 홀드(0) 신호 정확도 계산
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    print("\nClassification Report:\n", classification_report(y_true, y_pred, labels=[0, 1, 2], target_names=["Hold (0)", "Buy (1)", "Sell (2)"]))

    # Confusion Matrix 시각화
    plt.figure(figsize=(8, 6))
    plt.matshow(cm, cmap='Blues', fignum=1)
    plt.colorbar()
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix")
    plt.xticks([0, 1, 2], ["Hold (0)", "Buy (1)", "Sell (2)"])
    plt.yticks([0, 1, 2], ["Hold (0)", "Buy (1)", "Sell (2)"])

    for (i, j), val in np.ndenumerate(cm):
        plt.text(j, i, f'{val}', ha='center', va='center')

    plt.show()

# 테스트 데이터 준비 (예시)
# 실제 모델의 테스트 데이터 프레임(test_df)와 예측 값(y_pred)로 대체
test_df = pd.DataFrame({
    'Target': np.random.choice([0, 1, 2], size=100)  # 랜덤 타겟 데이터 (0: Hold, 1: Buy, 2: Sell)
})
y_test_pred = np.random.choice([0, 1, 2], size=100)  # 랜덤 예측 값 (모델 예측 결과로 대체)

# 정확도 시각화
visualize_accuracy(test_df, y_test_pred)